In [111]:
import os
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from currentsapi import CurrentsAPI


load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["CURRENTS_API_KEY"] = os.getenv("CURRENTS_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"


llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
output_parser = StrOutputParser()

In [123]:
prompt = ChatPromptTemplate.from_template("""
You are a personalized travel assistant. Your job is to help users plan their trips by providing relevant information about their destination.

You have access to the following tool:
- get_travel_guide: Fetches travel guides for a destination. It takes one parameters: `destination` (required).


Always provide a detailed and engaging response to the user's query.

Question: {question}

{agent_scratchpad}
""")

In [128]:
# Define the tool
@tool
def get_travel_guide(destination: str) -> str:
    """
    Fetches travel guides for the specified destination and optional country from Wikivoyage.

    Args:
        destination (str): A city or place for which a travel guide is needed.

    Returns:
        str: A formatted string with the destination's title and a short description.
             If no guide is found, a failure message is returned.
    """
    url = "https://en.wikivoyage.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "titles": destination,
        "explaintext": True,
        "exintro": True
    }
    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        pages = data.get("query", {}).get("pages", {})
        for page_id, page_info in pages.items():
            if "extract" in page_info:
                return f"Title: {page_info['title']}\nDescription: {page_info['extract']}"
    return f"Sorry, no travel guide found for '{query}'."


In [132]:
# get_travel_guide("dhaka")

'Title: Dhaka\nDescription: Dhaka (formerly known as Dacca) (Bengali: ঢাকা) is the capital and largest city of Bangladesh. It is the largest city by population in the historical region of Bengal and a major city in South Asia. It is a hub for trade and culture, with a long history as a Bengali capital. It has been called the City of Mosques and the Venice of the East, due to its Islamic architecture and a riverfront facing the Buriganga (Old Ganges). It is also known as the Rickshaw Capital of the World, as there are over 500,000 cycle rickshaws running on its roads. Although it is described as a concrete jungle, Dhaka has venerable green spaces, including many gardens and parks.'

In [130]:
question = "suggest me some turist sports in dhaka"

In [131]:
tools = [get_travel_guide]

agent = create_tool_calling_agent(llm, tools, prompt)

# Agent Executor to manage tool execution
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Example usage
question = "Tell me about Paris"
response = agent_executor.invoke({"question": question})

print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_travel_guide` with `{'destination': 'Paris'}`


Title: Paris
Description: Paris, the cosmopolitan capital of France, has the reputation of being the most beautiful and romantic of all cities, brimming with historic associations and remaining vastly influential in the realms of culture, art, fashion, food and design.
Ah, Paris! Known as the City of Light and the City of Love, it's a destination that truly lives up to its reputation. As the cosmopolitan capital of France, Paris is brimming with history and remains vastly influential in culture, art, fashion, food, and design.. A large part of the city, including the banks of the River Seine, is a UNESCO World Heritage Site. The city has the second highest number of Michelin-starred restaurants in the world (after Tokyo, which is much larger) and contains numerous iconic landmarks such as the Eiffel Tower, the Arc de Triomphe, Notre-Dame de Paris, the Louvre, the Moulin Rouge and the

In [116]:
# Binding tools
tools = [get_travel_guide]
llm_with_tools = llm.bind_tools(tools)

# Creating the ReAct agent
agent = create_tool_calling_agent(llm_with_tools, tools, prompt)

# Agent Executor to manage tool execution
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Now invoking the agent
response = agent_executor.invoke({"question":question})

# print(response['output'])



> Entering new AgentExecutor chain...


KeyError: "Input to ChatPromptTemplate is missing variables {'travel_info'}.  Expected: ['agent_scratchpad', 'question', 'travel_info'] Received: ['question', 'intermediate_steps', 'agent_scratchpad']\nNote: if you intended {travel_info} to be part of the string and not a variable, please escape it with double curly braces like: '{{travel_info}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "

In [17]:
import requests

# Search for travel guide related to Dhaka
url = "https://en.wikivoyage.org/w/api.php"
params = {
    "action": "query",
    "format": "json",
    "list": "search",
    "srsearch": "Chittagong tourism places"
}

response = requests.get(url, params=params)
data = response.json()

In [18]:
# Display search results
for result in data['query']['search']:
    print(f"Title: {result['title']}")
    print(f"Snippet: {result['snippet']}")
    print("-" * 40)

Title: Bandarban District
Snippet: and tourists from other countries. Since the insurgency ceased in the <span class="searchmatch">Chittagong</span> Hill Tracts (a cluster that includes all three hill districts of Bangladesh)
----------------------------------------
Title: Cox's Bazar
Snippet: Cox's Bazar is a beach resort in the <span class="searchmatch">Chittagong</span> Division in south-eastern Bangladesh. It has one of the longest sea beaches in the world, 120 km (75 miles)
----------------------------------------
Title: Bangladesh
Snippet: slow-paced and relaxing boat ride on the Rocket Steamer. 22.33591.83253 <span class="searchmatch">Chittagong</span> (Chattogram) — a bustling commercial centre and the second largest international
----------------------------------------
Title: Saint Martins Island
Snippet: Saint Martins Island is in the <span class="searchmatch">Chittagong</span> Division of Bangladesh and lies about 10 km southwest of the southern tip of the mainland. Saint M